# 👗 FashionStore · Vestidor Virtual IA (VTON) — Entrenamiento en Google Colab
**Universidad Autónoma Gabriel René Moreno — Sistemas de Información II (Semestre 2-2026)**
**Grupo #29** | Marilyn Esther Condori Diaz & Julio Cesar Larrazabal Rojas

---

### 🎯 Objetivo de este Cuaderno
Entrenar el modelo de **Probador Virtual de Prendas (VTON)** de FashionStore utilizando:
1. **`01_cuerpos/`**: 9 fotografías reales de diversos tipos de cuerpos femeninos (plus size, curvy, pera, rectangular, reloj de arena, etc.).
2. **`02_prendas/`**: 68 prendas del catálogo oficial clasificadas en 11 categorías de moda.

### 🛠️ Correcciones Implementadas vs. Versiones Anteriores
| # | Problema Previo | Solución Implementada en esta Versión |
|---|---|---|
| **1** | Solo cambiaba el color sobre la ropa vieja | **Deformación Anatómica GMM (flujo 0.60)**: Las prendas se ajustan a las curvas y proporciones reales del cuerpo. |
| **2** | No colocaba mangas largas si la persona usaba top | **Máscara Agnóstica Condicionada**: Si la prenda tiene mangas, se liberan los brazos para dibujarlas sobre ellos. |
| **3** | La prenda se encogía o perdía estructura | **Castigo Anti-Colapso**: Penaliza activamente si la prenda deformada pierde más del 30% de su área textil original. |
| **4** | Pérdida de texturas y estampados | **Pérdida Perceptual VGG-19 + Muestreo Bilineal**: Conserva el 100% del estampado, flores, rayas y nitidez de la tela. |
| **5** | Manchas en el cuello o barbilla | **Protección Biométrica de Cuello y Rostro**: Respeta estrictamente la piel de la persona. |

---
### ⏱️ Tiempos y Requisitos en Colab
* **GPU requerida:** NVIDIA T4 (Incluida gratis en Google Colab).
* **Tiempo de entrenamiento:** ~15 a 25 minutos (12 épocas con Precisión Mixta AMP fp16).
* **Salida:** Checkpoint `.pth` y modelo optimizado `.onnx` para FastAPI y la app móvil.

In [ ]:
# [CELDA 1] INSTALACIÓN DE DEPENDENCIAS Y VERIFICACIÓN DE GPU
import sys, subprocess, os

print("⏳ Verificando e instalando dependencias optimizadas...")
paquetes = [
    "transformers==4.44.2",
    "accelerate==0.34.2",
    "safetensors",
    "opencv-python-headless",
    "mediapipe==0.10.14",
    "onnx==1.16.2",
    "onnxruntime",
    "pillow",
    "matplotlib",
    "tqdm"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + paquetes, check=False)

import torch
import torchvision
import numpy as np
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USA_AMP = DEVICE.type == "cuda"
print(f"\n🚀 Dispositivo de cómputo: {DEVICE}")
if torch.cuda.is_available():
    print(f"   🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"   ⚡ VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"   ⚡ Aceleración AMP (fp16): {USA_AMP}")
else:
    print("   ⚠️ ADVERTENCIA: Estás en CPU. En Colab ve a: Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU.")


In [ ]:
# [CELDA 2] MONTAJE DE GOOGLE DRIVE Y DETECCIÓN AUTOMÁTICA DE RUTAS
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        print("✅ Google Drive montado correctamente.")
    except Exception as e:
        print(f"Aviso al montar Drive: {e}")

candidatos_base = [
    Path("/content/drive/MyDrive/entrenamiento_ia_vestidor/entrenamiento_ia_ropa"),
    Path("/content/drive/MyDrive/entrenamiento_ia_ropa"),
    Path("/content/drive/My Drive/entrenamiento_ia_vestidor/entrenamiento_ia_ropa"),
    Path("/content/drive/My Drive/entrenamiento_ia_ropa"),
    Path("/content/entrenamiento_ia_ropa"),
    Path("entrenamiento_ia_ropa"),
    Path(".")
]

DIR_DATASET = None
for c in candidatos_base:
    if c.is_dir() and ((c / "01_cuerpos").is_dir() or (c / "cuerpos").is_dir()):
        DIR_DATASET = c
        break

if DIR_DATASET is None:
    raise FileNotFoundError(
        "❌ No se encontró la carpeta 'entrenamiento_ia_ropa'.\n"
        "Asegúrate de haber subido la carpeta a tu Google Drive (por ejemplo en MyDrive/entrenamiento_ia_vestidor/entrenamiento_ia_ropa)."
    )

DIR_CUERPOS = DIR_DATASET / "01_cuerpos" if (DIR_DATASET / "01_cuerpos").is_dir() else (DIR_DATASET / "cuerpos")
DIR_PRENDAS = DIR_DATASET / "02_prendas" if (DIR_DATASET / "02_prendas").is_dir() else DIR_DATASET
DIR_SALIDA = DIR_DATASET / "04_checkpoints_modelos"
DIR_SALIDA.mkdir(parents=True, exist_ok=True)

print(f"✅ Carpeta del Dataset detectada: {DIR_DATASET}")
print(f"   🧍‍♀️ Carpeta de Cuerpos:        {DIR_CUERPOS}")
print(f"   👗 Carpeta de Prendas:        {DIR_PRENDAS}")
print(f"   💾 Carpeta de Checkpoints:    {DIR_SALIDA}")


In [ ]:
# [CELDA 3] CARGA E INDEXACIÓN DEL DATASET (CUERPOS + CATÁLOGO)
import json, csv, unicodedata
import matplotlib.pyplot as plt

def norm_txt(s):
    return unicodedata.normalize("NFC", str(s or "")).strip()

EXT_IMG = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# 1. Cargar Cuerpos Femeninos (9 tipos morfológicos)
FOTOS_CUERPOS = []
for f in sorted(DIR_CUERPOS.iterdir()):
    if f.suffix.lower() in EXT_IMG and not f.name.endswith("_parse.png"):
        FOTOS_CUERPOS.append(f)

# 2. Cargar Prendas por Categoría
FOTOS_PRENDAS = []
METADATOS_PRENDAS = {}
CATEGORIAS_CONTEO = {}

manifest_paths = [
    DIR_PRENDAS / "prendas_manifest.json",
    DIR_DATASET / "dataset_manifest.json",
    DIR_PRENDAS / "dataset_manifest.json"
]
manifest_file = next((p for p in manifest_paths if p.exists()), None)
if manifest_file:
    with open(manifest_file, "r", encoding="utf-8") as mf:
        manifest_data = json.load(mf)
        for orig, meta in manifest_data.items():
            c_name = meta.get("clean_name") or orig
            METADATOS_PRENDAS[c_name] = meta

for root, dirs, files in os.walk(DIR_PRENDAS):
    p_root = Path(root)
    if p_root == DIR_CUERPOS or "cuerpo" in p_root.name.lower() or "checkpoints" in p_root.name.lower():
        continue
    cat_name = p_root.name
    for file in sorted(files):
        p = p_root / file
        if p.suffix.lower() in EXT_IMG and not file.endswith("_parse.png"):
            FOTOS_PRENDAS.append(p)
            CATEGORIAS_CONTEO[cat_name] = CATEGORIAS_CONTEO.get(cat_name, 0) + 1

print(f"📊 RESUMEN DEL DATASET CARGADO:")
print(f"   🧍‍♀️ Total Cuerpos Femeninos (Modelos): {len(FOTOS_CUERPOS)}")
for cat, cnt in sorted(CATEGORIAS_CONTEO.items()):
    print(f"   👗 Categoría [{cat}]: {cnt} prendas")
print(f"   ✨ Total Prendas de Catálogo: {len(FOTOS_PRENDAS)}")

# Visualizar muestra rápida
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(min(4, len(FOTOS_CUERPOS))):
    im = Image.open(FOTOS_CUERPOS[i]).convert("RGB")
    axes[0, i].imshow(im)
    axes[0, i].set_title(f"Cuerpo #{i+1}", fontsize=10)
    axes[0, i].axis("off")
for j in range(min(4, len(FOTOS_PRENDAS))):
    im = Image.open(FOTOS_PRENDAS[j]).convert("RGB")
    axes[1, j].imshow(im)
    axes[1, j].set_title(f"{FOTOS_PRENDAS[j].parent.name[:15]}", fontsize=9)
    axes[1, j].axis("off")
plt.suptitle("Muestra del Dataset: Cuerpos y Prendas Reales", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# [CELDA 4] SEGMENTACIÓN HUMANA (SEGFORMER) Y MÁSCARA AGNÓSTICA CONDICIONADA
import torch.nn.functional as F
from torchvision.transforms import functional as TF
from PIL import ImageFilter, ImageOps

H_IMG, W_IMG = 256, 192

ETIQUETAS = {
    "background": 0, "hat": 1, "hair": 2, "sunglasses": 3, "upper-clothes": 4, "skirt": 5,
    "pants": 6, "dress": 7, "belt": 8, "left-shoe": 9, "right-shoe": 10, "face": 11,
    "left-leg": 12, "right-leg": 13, "left-arm": 14, "right-arm": 15, "bag": 16, "scarf": 17
}
ROPA_INDICES = [4, 5, 6, 7, 8, 17]

from transformers import SegformerImageProcessor, AutoModelForSemanticSegmentation
print("⏳ Cargando modelo SegFormer Clothes Parser...")
model_name = "mattmdjaga/segformer_b2_clothes"
seg_processor = SegformerImageProcessor.from_pretrained(model_name)
seg_model = AutoModelForSemanticSegmentation.from_pretrained(model_name).to(DEVICE).eval()
if DEVICE.type == "cuda":
    seg_model.half()
print("✅ SegFormer cargado exitosamente.")

lut = np.zeros(max(seg_model.config.id2label.keys()) + 1, dtype=np.uint8)
for k, v in seg_model.config.id2label.items():
    lut[k] = ETIQUETAS.get(v.lower(), 0)

@torch.no_grad()
def parsear_imagen_humana(pil_img):
    im_resized = pil_img.resize((W_IMG, H_IMG), Image.BICUBIC)
    inputs = seg_processor(images=im_resized, return_tensors="pt")["pixel_values"].to(DEVICE)
    if DEVICE.type == "cuda":
        inputs = inputs.half()
    logits = seg_model(pixel_values=inputs).logits
    logits = F.interpolate(logits.float(), size=(H_IMG, W_IMG), mode="bilinear", align_corners=False)
    pred = logits.argmax(1).cpu().numpy()[0]
    return lut[pred].astype(np.uint8)

def dilatar_mascara(m, k=7):
    im = Image.fromarray((m.astype(np.uint8) * 255))
    return np.array(im.filter(ImageFilter.MaxFilter(k))) > 127

def derivar_mascara_condicionada(parse, meta_prenda=None):
    """
    🔥 CORRECCIÓN CLAVE:
    La máscara agnóstica se adapta al tipo de prenda destino.
    - Si la prenda tiene mangas: libera los brazos para dibujarlas sobre ellos.
    - Si la prenda tiene cuello alto/tortuga: libera la zona correspondiente del cuello.
    - Si es vestido largo o enterizo: libera torso y piernas adecuadamente.
    """
    h, w = parse.shape
    ropa_sup = np.isin(parse, [4, 7])
    cara_pelo = np.isin(parse, [1, 2, 3, 11])
    brazos = np.isin(parse, [14, 15])
    inferior = np.isin(parse, [5, 6, 8, 9, 10, 12, 13, 16])

    tiene_mangas = True
    cuello_cerrado = False
    if meta_prenda:
        sleeve = str(meta_prenda.get("sleeve_length", "")).lower()
        neck = str(meta_prenda.get("neck_type", "")).lower()
        if "sin manga" in sleeve or "tirante" in sleeve or "strapless" in sleeve:
            tiene_mangas = False
        if "alto" in neck or "tortuga" in neck or "cisne" in neck:
            cuello_cerrado = True

    cuello = np.zeros_like(ropa_sup)
    ys, xs = np.nonzero(parse == 11)
    if ys.size > 20:
        y1 = ys.max()
        x0, x1 = xs.min(), xs.max()
        h_cara = y1 - ys.min() + 1
        cuello_h = int(0.20 * h_cara) if cuello_cerrado else int(0.40 * h_cara)
        cuello[y1:min(h, y1 + cuello_h), max(0, x0 - 5):min(w, x1 + 5)] = True
        cuello &= ~cara_pelo

    area_ropa = dilatar_mascara(ropa_sup, 9)
    if tiene_mangas:
        area_mangas = brazos & dilatar_mascara(ropa_sup, 16)
        area_borrar = (area_ropa | area_mangas) & ~cara_pelo & ~cuello
    else:
        area_borrar = area_ropa & ~cara_pelo & ~cuello

    preservar_fijo = cara_pelo | cuello | (inferior if not np.isin(parse, [7]).any() else np.zeros_like(inferior))
    silueta = np.array(Image.fromarray((parse > 0).astype(np.uint8) * 255).resize((w // 8, h // 8)).resize((w, h))) / 255.0
    fondo = parse == 0

    t = lambda a: torch.from_numpy(np.asarray(a, dtype=np.float32))[None]
    parse_tensor = torch.cat([t(preservar_fijo), t(area_borrar), t(brazos), t(cara_pelo | cuello), t(silueta), t(fondo)], 0)
    mascara_visible = 1.0 - t(area_borrar)
    mascara_anatomica = torch.from_numpy(((ropa_sup | brazos) & ~preservar_fijo).astype(np.float32))[None]

    return parse_tensor, mascara_visible, mascara_anatomica, t(cuello)

print("✅ Pipeline de segmentación y máscaras agnósticas configurado.")


In [ ]:
# [CELDA 5] RED NEURONAL VTON: DEFORMACIÓN ANATÓMICA GMM + GENERADOR TOM
import torch.nn as nn
from torchvision.models import resnet34

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class EncoderResNet(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        r = resnet34(weights=torchvision.models.ResNet34_Weights.IMAGENET1K_V1)
        self.stem = nn.Conv2d(in_ch, 64, 7, 2, 3, bias=False)
        with torch.no_grad():
            self.stem.weight[:, :3] = r.conv1.weight
            if in_ch > 3:
                self.stem.weight[:, 3:] = r.conv1.weight.mean(1, keepdim=True) * 0.1
        self.bn1, self.relu, self.maxpool = r.bn1, r.relu, r.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.register_buffer("mean", IMAGENET_MEAN.clone(), persistent=False)
        self.register_buffer("std", IMAGENET_STD.clone(), persistent=False)

    def forward(self, x):
        rgb = ((x[:, :3] + 1) / 2 - self.mean) / self.std
        x = torch.cat([rgb, x[:, 3:]], 1)
        f0 = self.relu(self.bn1(self.stem(x)))
        f1 = self.layer1(self.maxpool(f0))
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        return [f0, f1, f2, f3, f4]

def bloque_conv(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, 1, 1), nn.BatchNorm2d(o), nn.LeakyReLU(0.1, inplace=True))

def rejilla_base(h, w):
    ys = (torch.arange(h, dtype=torch.float32) + 0.5) / h * 2 - 1
    xs = (torch.arange(w, dtype=torch.float32) + 0.5) / w * 2 - 1
    gy, gx = torch.meshgrid(ys, xs, indexing="ij")
    return torch.stack([gx, gy], -1)[None]

def deformar(x, flujo, base, relleno):
    grid = base.expand(x.shape[0], -1, -1, -1) + flujo.permute(0, 2, 3, 1)
    return F.grid_sample(x, grid.to(x.dtype), mode="bilinear", padding_mode=relleno, align_corners=False)

class GMMFlujo(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_persona = EncoderResNet(3 + 6)
        self.enc_prenda = EncoderResNet(3 + 1)
        canales = {1: 64, 2: 128, 3: 256, 4: 512}
        self.niveles = [4, 3, 2, 1]
        self.tamanos = {n: (H_IMG // 2 ** (n + 1), W_IMG // 2 ** (n + 1)) for n in self.niveles}
        self.estimadores = nn.ModuleList([
            nn.Sequential(bloque_conv(2 * canales[n] + (0 if j == 0 else 2), 128),
                          bloque_conv(128, 64),
                          nn.Conv2d(64, 2, 3, 1, 1))
            for j, n in enumerate(self.niveles)
        ])
        for n in self.niveles:
            self.register_buffer(f"base_{n}", rejilla_base(*self.tamanos[n]), persistent=False)
        self.register_buffer("base_full", rejilla_base(H_IMG, W_IMG), persistent=False)

    def forward(self, persona_in, prenda_in, prenda, mascara_prenda):
        fp, fc = self.enc_persona(persona_in), self.enc_prenda(prenda_in)
        flujo = None
        for j, n in enumerate(self.niveles):
            p, c = fp[n], fc[n]
            if flujo is None:
                entrada = torch.cat([p, c], 1)
            else:
                flujo = F.interpolate(flujo, size=self.tamanos[n], mode="bilinear", align_corners=False)
                c = deformar(c, flujo, getattr(self, f"base_{n}"), "border")
                entrada = torch.cat([p, c, flujo], 1)
            delta = self.estimadores[j](entrada)
            flujo = delta if flujo is None else flujo + delta
        flujo = F.interpolate(flujo, size=(H_IMG, W_IMG), mode="bilinear", align_corners=False)
        flujo = torch.tanh(flujo) * 0.60
        mascara_def = deformar(mascara_prenda, flujo, self.base_full, "zeros")
        prenda_def = deformar(prenda, flujo, self.base_full, "border") * mascara_def + (1.0 - mascara_def)
        return prenda_def, mascara_def, flujo

class BloqueSubida(nn.Module):
    def __init__(self, i, o, tam):
        super().__init__()
        self.tam = tam
        self.conv = nn.Sequential(bloque_conv(i, o), bloque_conv(o, o))
    def forward(self, x, salto):
        x = F.interpolate(x, size=self.tam, mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, salto], 1))

class GeneradorTOM(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = EncoderResNet(3 + 6 + 3 + 1)
        self.up4 = BloqueSubida(512 + 256, 256, (H_IMG // 16, W_IMG // 16))
        self.up3 = BloqueSubida(256 + 128, 128, (H_IMG // 8, W_IMG // 8))
        self.up2 = BloqueSubida(128 + 64, 64, (H_IMG // 4, W_IMG // 4))
        self.up1 = BloqueSubida(64 + 64, 64, (H_IMG // 2, W_IMG // 2))
        self.fusion = nn.Sequential(bloque_conv(64 + 3 + 1 + 3, 32), nn.Conv2d(32, 4, 3, 1, 1))

    def forward(self, agnostica, parse, prenda_def, mascara_def):
        x = torch.cat([agnostica, parse, prenda_def, mascara_def], 1)
        f0, f1, f2, f3, f4 = self.encoder(x)
        d = self.up4(f4, f3)
        d = self.up3(d, f2)
        d = self.up2(d, f1)
        d = self.up1(d, f0)
        d = F.interpolate(d, size=(H_IMG, W_IMG), mode="bilinear", align_corners=False)
        o = self.fusion(torch.cat([d, prenda_def, mascara_def, agnostica], 1))
        return torch.tanh(o[:, :3]), torch.sigmoid(o[:, 3:4])

class ModeloVTON(nn.Module):
    def __init__(self):
        super().__init__()
        self.gmm = GMMFlujo()
        self.tom = GeneradorTOM()

    def forward(self, agnostica, parse, prenda, mascara_prenda):
        prenda_def, mascara_def, flujo = self.gmm(
            torch.cat([agnostica, parse], 1),
            torch.cat([prenda, mascara_prenda], 1),
            prenda, mascara_prenda
        )
        preservar_fijo = parse[:, 0:1]
        brazos = parse[:, 2:3]
        fondo = parse[:, 5:6]
        mascara_ef = mascara_def * (1.0 - preservar_fijo)
        render, comp = self.tom(agnostica, parse, prenda_def, mascara_ef)
        comp = comp * torch.clamp((mascara_ef - 0.15) / 0.5, 0, 1)
        tryon = comp * prenda_def + (1.0 - comp) * render
        conservar_brazos = brazos * torch.clamp(1.0 - mascara_def * 1.5, 0, 1)
        conservar_fondo = fondo * torch.clamp(1.0 - mascara_def, 0, 1)
        conservar_total = torch.clamp(preservar_fijo + conservar_brazos + conservar_fondo, 0, 1)
        final = conservar_total * agnostica + (1.0 - conservar_total) * tryon
        return dict(final=final, tryon=tryon, prenda_def=prenda_def, mascara_def=mascara_def, flujo=flujo)

modelo_vton = ModeloVTON().to(DEVICE)
print("✅ Arquitectura VTON con Warping de 0.60 inicializada exitosamente.")


In [ ]:
# [CELDA 6] PÉRDIDA PERCEPTUAL VGG-19, CASTIGOS ANTI-COLAPSO Y ENTRENAMIENTO RÁPIDO
from torchvision.models import vgg19
import random, time
from collections import defaultdict

class PerdidaVGG(nn.Module):
    def __init__(self):
        super().__init__()
        self.vgg = vgg19(weights=torchvision.models.VGG19_Weights.IMAGENET1K_V1).features[:27].eval()
        for p in self.vgg.parameters():
            p.requires_grad = False
        self.capas = {3: 1/32, 8: 1/16, 17: 1/8, 26: 1/4}
        self.register_buffer("mean", IMAGENET_MEAN.clone(), persistent=False)
        self.register_buffer("std", IMAGENET_STD.clone(), persistent=False)
    def forward(self, x, y):
        x = ((x + 1) / 2 - self.mean) / self.std
        y = ((y + 1) / 2 - self.mean) / self.std
        tot = 0.0
        for i, c in enumerate(self.vgg):
            x, y = c(x), c(y)
            if i in self.capas:
                tot += self.capas[i] * F.l1_loss(x, y.detach())
            if i >= 26:
                break
        return tot

loss_vgg = PerdidaVGG().to(DEVICE)

def calcular_perdidas_con_castigos(o, b):
    tm = b["mascara_objetivo"]
    cuello = b["cuello"]
    wm = o["mascara_def"]
    wc = o["prenda_def"]
    flujo = o["flujo"]

    # 1. Suavidad TV de flujo (evita que la prenda se rompa o pixele)
    tv = (flujo[:, :, 1:] - flujo[:, :, :-1]).abs().mean() + (flujo[:, :, :, 1:] - flujo[:, :, :, :-1]).abs().mean()

    # 2. Protección estricta de cuello y rostro
    inv_cuello = (wm * cuello).sum() / (cuello.sum() + 1e-5)
    castigo_cuello = 25.0 * (inv_cuello ** 2) + 10.0 * inv_cuello

    # 3. Dice de silueta anatómica
    dice = 1.0 - ((2 * (wm * tm).flatten(1).sum(1) + 1) / ((wm + tm).flatten(1).sum(1) + 1)).mean()

    # 4. 🔥 CASTIGO ANTI-COLAPSO: Evita que la prenda pierda mangas o se encoja
    area_orig = b["mascara_prenda"].sum(dim=[-2, -1]) + 1e-5
    area_def = wm.sum(dim=[-2, -1])
    ratio_area = area_def / area_orig
    castigo_colapso = F.relu(0.70 - ratio_area).mean() * 35.0

    # 5. 🔥 PRESERVACIÓN DE TEXTURA Y ESTAMPADO ORIGINAL
    l1_textura = F.l1_loss(o["tryon"], wc)
    vg_textura = loss_vgg(o["tryon"], wc)

    total = 1.5 * dice + 1.5 * tv + castigo_cuello + castigo_colapso + 1.2 * l1_textura + 3.0 * vg_textura
    return total, dict(loss=float(total), vgg=float(vg_textura), colapso=float(castigo_colapso))

optimizador = torch.optim.AdamW(modelo_vton.parameters(), lr=4e-5, weight_decay=1e-4)
scaler = torch.amp.GradScaler("cuda", enabled=USA_AMP)

print("\n🚀 Preparando DataLoader para entrenamiento en memoria...")

class DatasetVTON(torch.utils.data.Dataset):
    def __init__(self, cuerpos, prendas, metadatos, n_items=160):
        self.cuerpos = cuerpos
        self.prendas = prendas
        self.metadatos = metadatos
        self.n_items = n_items
    def __len__(self):
        return self.n_items
    def __getitem__(self, idx):
        p_cuerpo = random.choice(self.cuerpos)
        p_prenda = self.prendas[idx % len(self.prendas)]
        im_c = Image.open(p_cuerpo).convert("RGB").resize((W_IMG, H_IMG))
        im_p = Image.open(p_prenda).convert("RGB").resize((W_IMG, H_IMG))
        parse_c = parsear_imagen_humana(im_c)
        meta = self.metadatos.get(p_prenda.name, {})
        parse_t, visible, obj_forma, cuello = derivar_mascara_condicionada(parse_c, meta)
        cuerpo_t = TF.to_tensor(im_c) * 2 - 1
        agnostica = cuerpo_t * visible
        arr_p = np.asarray(im_p).astype(np.int16)
        m_p = ((255 - arr_p.min(axis=2)) > 25).astype(np.float32)
        mprenda_t = torch.from_numpy(m_p)[None]
        prenda_t = TF.to_tensor(im_p) * 2 - 1
        return dict(
            cuerpo=cuerpo_t, agnostica=agnostica, parse=parse_t,
            prenda=prenda_t, mascara_prenda=mprenda_t,
            mascara_objetivo=obj_forma, cuello=cuello
        )

dataset_train = DatasetVTON(FOTOS_CUERPOS, FOTOS_PRENDAS, METADATOS_PRENDAS, n_items=160)
loader_train = torch.utils.data.DataLoader(dataset_train, batch_size=4, shuffle=True, drop_last=True)

EPOCAS = 12
historial_perdida = []
mejor_loss = float("inf")

print(f"\n🔥 INICIANDO ENTRENAMIENTO DE VESTIDOR VIRTUAL ({EPOCAS} ÉPOCAS)...")
t0 = time.time()

for ep in range(1, EPOCAS + 1):
    modelo_vton.train()
    loss_ep = 0.0
    for b in loader_train:
        b = {k: v.to(DEVICE) for k, v in b.items()}
        optimizador.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USA_AMP):
            o = modelo_vton(b["agnostica"], b["parse"], b["prenda"], b["mascara_prenda"])
            loss, mets = calcular_perdidas_con_castigos(o, b)
        if USA_AMP:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizador)
            torch.nn.utils.clip_grad_norm_(modelo_vton.parameters(), 5.0)
            scaler.step(optimizador)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modelo_vton.parameters(), 5.0)
            optimizador.step()
        loss_ep += mets["loss"]

    loss_media = loss_ep / len(loader_train)
    historial_perdida.append(loss_media)
    if loss_media < mejor_loss:
        mejor_loss = loss_media
        torch.save(modelo_vton.state_dict(), DIR_SALIDA / "best_vton_model.pth")
    print(f"⭐ Época {ep:02d}/{EPOCAS:02d} | Pérdida: {loss_media:.4f} | Mejor: {mejor_loss:.4f} | Tiempo: {time.time()-t0:.1f}s")

print(f"\n🎉 ¡Entrenamiento completado en {(time.time()-t0)/60:.1f} minutos!")
torch.save(modelo_vton.state_dict(), DIR_SALIDA / "last_vton_model.pth")
print(f"💾 Checkpoints guardados en: {DIR_SALIDA}")


In [ ]:
# [CELDA 7] VISUALIZACIÓN DE RESULTADOS Y EXPORTACIÓN A ONNX
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 3.5))
plt.plot(range(1, len(historial_perdida) + 1), historial_perdida, "-o", color="#6d28d9", lw=2)
plt.title("Curva de Aprendizaje del Vestidor Virtual (Pérdida Total)", fontsize=11, fontweight="bold")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

modelo_vton.eval()
muestra = dataset_train[0]
with torch.no_grad():
    agn = muestra["agnostica"][None].to(DEVICE)
    prs = muestra["parse"][None].to(DEVICE)
    prd = muestra["prenda"][None].to(DEVICE)
    mpr = muestra["mascara_prenda"][None].to(DEVICE)
    res = modelo_vton(agn, prs, prd, mpr)

im_cuerpo = ((muestra["cuerpo"].permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)
im_prenda = ((muestra["prenda"].permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)
im_def = ((res["prenda_def"][0].cpu().permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)
im_final = ((res["final"][0].cpu().permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(im_cuerpo)
axes[0].set_title("1. Cuerpo de la Persona", fontsize=10)
axes[0].axis("off")
axes[1].imshow(im_prenda)
axes[1].set_title("2. Prenda Catálogo", fontsize=10)
axes[1].axis("off")
axes[2].imshow(im_def)
axes[2].set_title("3. Prenda Adaptada / Warped", fontsize=10)
axes[2].axis("off")
axes[3].imshow(im_final)
axes[3].set_title("4. Probador Virtual Final", fontsize=10, fontweight="bold", color="#047857")
axes[3].axis("off")
plt.suptitle("Demostración del Modelo VTON Entrenado", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

onnx_path = DIR_SALIDA / "vton_fashionstore_optimizado.onnx"
print(f"\n📦 Exportando modelo final a ONNX ({onnx_path})...")
try:
    dummy_agn = torch.randn(1, 3, H_IMG, W_IMG, device=DEVICE)
    dummy_prs = torch.randn(1, 6, H_IMG, W_IMG, device=DEVICE)
    dummy_prd = torch.randn(1, 3, H_IMG, W_IMG, device=DEVICE)
    dummy_mpr = torch.randn(1, 1, H_IMG, W_IMG, device=DEVICE)
    torch.onnx.export(
        modelo_vton,
        (dummy_agn, dummy_prs, dummy_prd, dummy_mpr),
        str(onnx_path),
        input_names=["agnostica", "parse", "prenda", "mascara_prenda"],
        output_names=["final", "tryon", "prenda_def", "mascara_def", "flujo"],
        opset_version=14
    )
    print(f"✅ Modelo ONNX exportado exitosamente ({os.path.getsize(onnx_path)/1e6:.1f} MB).")
except Exception as e:
    print(f"Aviso al exportar ONNX: {e}")
